# 🧪 Lab 07: Make the Compiler Flinch

Welcome to the generated-code triage bay. WholeStageCodegen is useful because it specializes a known plan, but specialization can become its own engineering problem when the plan becomes enormous.

**Mission Objective:** build progressively larger `CASE WHEN` expressions, inspect their generated-code shape and size, and then execute the same aggregations. We are looking for growth, helper-method behavior, and changes in codegen eligibility—not a universal branch-count failure threshold.

**Deterministic Guardrail:** every query reads the same generated range and returns one checksum. The branch counts are intentionally bounded because the exact failure point depends on Spark, the JVM, the expression, and the surrounding plan.


### Step 1: Define the diagnostic session
Adaptive execution is disabled so the codegen-stage shape remains comparable across expression sizes.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import io
from contextlib import redirect_stdout

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-07-make-the-compiler-flinch")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:25:48 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:25:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 06:25:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Generate progressively larger expressions
Each query maps `id` through a searched `CASE` expression and immediately aggregates the result. The aggregation forces Spark to compute the generated value instead of pruning it away.


In [2]:
def build_case_query(branches):
    cases = " ".join(f"WHEN id = {i} THEN {i}" for i in range(branches))
    case_sql = f"CASE {cases} ELSE -1 END AS result"
    return (spark.range(0, 100_000)
        .selectExpr(case_sql)
        .agg(F.sum("result").alias("checksum")))

def inspect_codegen(branches, show_excerpt=False):
    query = build_case_query(branches)
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        query.explain("codegen")
    generated = buffer.getvalue()
    lines = generated.splitlines()
    landmarks = [line.strip() for line in lines if (
        "GeneratedIteratorForCodegenStage" in line
        or "processNext" in line
        or "maxMethodCodeSize" in line
        or "WholeStageCodegen" in line
    )]
    print(f"branches={branches}, generated_output_chars={len(generated)}, lines={len(lines)}")
    print("landmarks:", landmarks[:8])
    if show_excerpt:
        print("--- generated-code excerpt ---")
        print("\n".join(lines[:35]))
    return query, generated


### Step 3: Inspect the codegen shape before executing
Start with a small expression that is easy to recognize, then compare larger generated programs by their captured output size and landmarks. The exact bytecode limit is not the target; the changing shape is the evidence.


In [3]:
inspections = {}
for branches in (10, 100, 500):
    inspections[branches] = inspect_codegen(branches, show_excerpt=(branches == 10))


branches=10, generated_output_chars=19592, lines=421
landmarks: ['Found 2 WholeStageCodegen subtrees.', '== Subtree 1 / 2 (maxMethodCodeSize:364; maxConstantPoolSize:244(0.37% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage1(references);', '/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {', '/* 022 */   public GeneratedIteratorForCodegenStage1(Object[] references) {', '/* 251 */   protected void processNext() throws java.io.IOException {', '== Subtree 2 / 2 (maxMethodCodeSize:149; maxConstantPoolSize:158(0.24% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage2(references);']
--- generated-code excerpt ---
Found 2 WholeStageCodegen subtrees.
== Subtree 1 / 2 (maxMethodCodeSize:364; maxConstantPoolSize:244(0.37% used); numInnerClasses:0) ==
*(1) HashAggregate(keys=[], functions=[partial_sum(result#1)], output=[sum#6L])
+- *(1) Project [CASE WHE

branches=100, generated_output_chars=55011, lines=1141
landmarks: ['Found 2 WholeStageCodegen subtrees.', '== Subtree 1 / 2 (maxMethodCodeSize:3424; maxConstantPoolSize:424(0.65% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage1(references);', '/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {', '/* 022 */   public GeneratedIteratorForCodegenStage1(Object[] references) {', '/* 971 */   protected void processNext() throws java.io.IOException {', '== Subtree 2 / 2 (maxMethodCodeSize:149; maxConstantPoolSize:158(0.24% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage2(references);']


branches=500, generated_output_chars=219964, lines=4341
landmarks: ['Found 2 WholeStageCodegen subtrees.', '== Subtree 1 / 2 (maxMethodCodeSize:18884; maxConstantPoolSize:1224(1.87% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage1(references);', '/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {', '/* 022 */   public GeneratedIteratorForCodegenStage1(Object[] references) {', '/* 4171 */   protected void processNext() throws java.io.IOException {', '== Subtree 2 / 2 (maxMethodCodeSize:149; maxConstantPoolSize:158(0.24% used); numInnerClasses:0) ==', '/* 002 */   return new GeneratedIteratorForCodegenStage2(references);']


### Step 4: Execute each expression and compare correctness
Now materialize the checksum for every size. A larger generated program is not automatically wrong; the result must remain consistent with the same deterministic input and branch definition.


In [4]:
ROWS = 100_000

def expected_checksum(branches):
    matched = branches * (branches - 1) // 2
    unmatched = ROWS - branches
    return matched - unmatched

for branches, (query, generated) in inspections.items():
    checksum = query.collect()[0]["checksum"]
    expected = expected_checksum(branches)

    print(
        f"branches={branches}, "
        f"checksum={checksum}, expected={expected}, "
        f"correct={checksum == expected}"
    )

    assert checksum == expected

branches=10, checksum=-99945, expected=-99945, correct=True
branches=100, checksum=-94950, expected=-94950, correct=True


branches=500, checksum=25250, expected=25250, correct=True


### Step 5: Look for a stage-shape change
The experiment is successful if it gives us evidence about generated-program growth and whether Spark continues to admit the expression into a codegen stage. If codegen eligibility changes or Spark falls back, that is useful evidence—not a correctness failure.


In [5]:
for branches, (_, generated) in inspections.items():
    print(f"branches={branches}: WholeStageCodegen={'WholeStageCodegen' in generated}, processNext={'processNext' in generated}, generated_chars={len(generated)}")
print("Interpretation: generated-code size and eligibility are workload- and version-dependent.")


branches=10: WholeStageCodegen=True, processNext=True, generated_chars=19592
branches=100: WholeStageCodegen=True, processNext=True, generated_chars=55011
branches=500: WholeStageCodegen=True, processNext=True, generated_chars=219964
Interpretation: generated-code size and eligibility are workload- and version-dependent.


# 📊 Post-Lab Analysis: When Generated Code Becomes the Problem

This lab increased expression complexity while holding the input and aggregation shape constant. The important evidence is not a magical number of `WHEN` clauses; it is that the generated program itself grows and can eventually become subject to code-size, method-size, or stage-eligibility constraints.

### 1. Specialization Has a Size

The generated-code output grows as more branches become part of the same expression. Spark is specializing the plan, but the specialized program must still be represented, compiled, loaded, and executed by the JVM.

### 2. Fallback Protects Correctness

Spark has fallback and code-splitting mechanisms for generated code that becomes unsuitable. This particular run may not trigger them; it shows the generated program growing toward the constraints those mechanisms protect against. Correctness therefore does not prove that the generated path remained unchanged.

### 3. Do Not Worship the Threshold

The exact point where a particular expression becomes too large depends on Spark version, JVM, expression shape, and surrounding operators. This lab records evidence from this runtime instead of pretending that one branch count is universal.

### 4. The Better Fix Is Often a Smaller Plan

Before raising limits, consider pruning unused columns, simplifying generated expressions, splitting an enormous transformation, or replacing thousands of literal branches with data. Sometimes Spark protects itself from the Java that the query asked it to write.
